# 🍎 AlphaApple Training V3 (Colab)

**V3 개선사항**:
- ✅ **Multiple Rollouts**: 각 보드당 여러 번 플레이, 최고 성능 선택
- ✅ **Temperature Sampling**: 다양한 전략 탐색
- ✅ **더 많은 데이터**: 500 → 1000 에피소드
- ✅ **Early Stopping**: Validation loss 기반 조기 종료
- ✅ **Output 요약**: Claude Code용 결과 요약 출력

**목표**: 사람 최고(130개, 76.5%)에 도달

**핵심 아이디어**: Greedy 전략의 문제는 "항상 작은 것"만 선택한다는 것. Temperature sampling으로 다양한 전략을 시도하고, 각 보드에서 best를 선택!

## 🔧 Setup

In [ ]:
# Colab 환경 확인
try:
    import google.colab
    IN_COLAB = True
    print("✅ Running in Colab")
except:
    IN_COLAB = False
    print("❌ Not in Colab")

# GPU 확인
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# GitHub에서 코드 가져오기
if IN_COLAB:
    !git clone https://github.com/kbsooo/AlphaApple.git
    %cd AlphaApple
    !git checkout claude/train-colab-v2-results-011CUyksmu43qWNznViHZEKp
else:
    import os
    os.chdir('/home/user/AlphaApple')

In [ ]:
# 의존성 설치
!pip install -q gymnasium numpy torch tqdm

## 🎯 1. Multiple Rollouts Expert (핵심 개선!)

**Greedy의 문제점**: 
- 항상 "작은 것 우선" → 다른 좋은 전략 놓칠 수 있음
- 단일 경로만 탐색

**Multiple Rollouts + Temperature Sampling**:
- 각 보드를 **5번 플레이** (서로 다른 temperature)
- 가장 좋은 결과만 학습 데이터로 사용
- Temperature: 0.5 (보수적) ~ 2.0 (공격적) 범위
- 간단하고 효과적!

In [ ]:
import sys
import numpy as np
import pickle
from tqdm.notebook import tqdm

sys.path.insert(0, '.')

from envs.fruitbox_env import FruitBoxEnv, FruitBoxConfig
from envs.backward_generator import BackwardBoardGenerator
from envs.autoregressive_wrapper import make_autoregressive_env

In [ ]:
def play_with_temperature(initial_board, temperature=1.0, strategy='small_first'):
    """
    Temperature 기반 샘플링으로 플레이
    
    Args:
        initial_board: 초기 보드
        temperature: 높을수록 랜덤, 낮을수록 greedy
        strategy: 'small_first' (작은 것 우선) 또는 'large_first' (큰 것 우선)
    
    Returns:
        history: [(obs, action, reward, mask), ...]
        total_reward: 최종 보상
    """
    wrapped_env = make_autoregressive_env(rows=10, cols=17)
    env = wrapped_env.env
    env.board = initial_board.copy().astype(np.int16)
    obs = initial_board.clip(0, 9).astype(np.int8)
    
    history = []
    total_reward = 0
    steps = 0
    
    while steps < 500:
        # 합법 행동들
        legal = env.legal_actions()
        if len(legal) == 0:
            break
        
        # Action mask
        autoregressive_masks = wrapped_env.get_autoregressive_masks()
        
        # 크기 계산
        sizes = np.array([(env.rects[a][2]-env.rects[a][0]+1) * (env.rects[a][3]-env.rects[a][1]+1) 
                          for a in legal])
        
        # Strategy에 따라 정렬
        if strategy == 'small_first':
            scores = -sizes  # 작은 것이 높은 점수
        else:
            scores = sizes  # 큰 것이 높은 점수
        
        # Temperature scaling
        scores = scores / temperature
        
        # Softmax로 확률 계산
        exp_scores = np.exp(scores - np.max(scores))  # numerical stability
        probs = exp_scores / np.sum(exp_scores)
        
        # 샘플링
        action_idx = np.random.choice(len(legal), p=probs)
        action = legal[action_idx]
        
        r1, c1, r2, c2 = env.rects[action]
        
        # 기록
        history.append((obs.copy(), (r1, c1, r2, c2), 0, autoregressive_masks))
        
        # Step
        obs, reward, terminated, truncated, info = wrapped_env.step_with_coords(r1, c1, r2, c2)
        
        history[-1] = (history[-1][0], history[-1][1], reward, history[-1][3])
        total_reward += reward
        steps += 1
        
        if terminated or truncated:
            break
    
    return history, total_reward


def collect_expert_data_multiple_rollouts(
    n_episodes=1000,
    target_coverage=0.95,
    n_rollouts=5
):
    """
    Multiple Rollouts로 고품질 expert 데이터 수집
    
    각 보드에서 n_rollouts번 플레이하고 가장 좋은 것만 저장
    """
    episodes = []
    total_rewards = []
    
    for i in tqdm(range(n_episodes), desc="Collecting with Multiple Rollouts"):
        # 역방향 생성
        generator = BackwardBoardGenerator(rows=10, cols=17, seed=i)
        board, solution = generator.generate(target_coverage=target_coverage)
        
        # 여러 번 플레이
        best_history = None
        best_reward = -float('inf')
        
        # 다양한 temperature와 strategy 시도
        configs = [
            (0.5, 'small_first'),   # 매우 보수적
            (1.0, 'small_first'),   # 중간
            (1.5, 'small_first'),   # 탐색적
            (1.0, 'large_first'),   # 큰 것 우선
            (0.7, 'small_first'),   # 균형
        ]
        
        for temp, strat in configs[:n_rollouts]:
            history, reward = play_with_temperature(board, temperature=temp, strategy=strat)
            
            if reward > best_reward:
                best_reward = reward
                best_history = history
        
        if best_history is None or len(best_history) == 0:
            continue
        
        # 데이터 저장
        observations = []
        actions = []
        rewards = []
        masks = []
        
        for obs, action, reward, mask in best_history:
            observations.append(obs)
            actions.append(action)
            rewards.append(reward)
            masks.append(mask)
        
        episodes.append({
            'observations': np.array(observations),
            'actions': np.array(actions),
            'rewards': np.array(rewards),
            'masks': masks,
            'total_reward': best_reward,
            'steps': len(best_history),
            'seed': i,
        })
        
        total_rewards.append(best_reward)
    
    print(f"\n=== 수집 완료 ===")
    print(f"에피소드 수: {len(episodes)}")
    print(f"평균 보상: {np.mean(total_rewards):.1f} ± {np.std(total_rewards):.1f}")
    print(f"최대 보상: {max(total_rewards):.0f}")
    print(f"최소 보상: {min(total_rewards):.0f}")
    print(f"총 transition: {sum(ep['steps'] for ep in episodes)}")
    
    return episodes

In [ ]:
# 데이터 수집 (1000 episodes × 5 rollouts, ~15-20분 예상)
# 더 빠른 테스트를 원하면 n_episodes=500, n_rollouts=3으로 조정
expert_data = collect_expert_data_multiple_rollouts(
    n_episodes=1000,
    target_coverage=0.95,
    n_rollouts=5
)

# 저장
with open('expert_data_95pct_v3.pkl', 'wb') as f:
    pickle.dump(expert_data, f)

print("✅ 데이터 저장 완료")

## 🧠 2. 모델 학습 (Early Stopping)

In [ ]:
from models.lightweight_policy import LightweightPolicy

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 모델 생성
policy = LightweightPolicy(rows=10, cols=17, latent_dim=128)
policy = policy.to(device)

print(f"파라미터 수: {sum(p.numel() for p in policy.parameters()):,}")
print(f"Device: {device}")

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

class ExpertDatasetWithMasks(Dataset):
    def __init__(self, episodes):
        self.data = []
        
        for ep in episodes:
            for t in range(len(ep['observations'])):
                self.data.append({
                    'obs': ep['observations'][t],
                    'action': ep['actions'][t],
                    'mask': ep['masks'][t]
                })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        obs = torch.from_numpy(item['obs']).float().unsqueeze(0)  # (1, 10, 17)
        act = torch.from_numpy(np.array(item['action'])).long()  # (4,)
        
        masks = item['mask']
        masks_torch = {
            'r1_mask': torch.from_numpy(masks['r1_mask']),
            'c1_masks': torch.from_numpy(masks['c1_masks']),
            'r2_masks': torch.from_numpy(masks['r2_masks']),
            'c2_masks': torch.from_numpy(masks['c2_masks'])
        }
        
        return obs, act, masks_torch

# 데이터셋 생성
dataset = ExpertDatasetWithMasks(expert_data)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# Behavior Cloning with Early Stopping
optimizer = optim.Adam(policy.parameters(), lr=3e-4)
n_epochs = 100
best_val_loss = float('inf')
patience = 10  # Early stopping patience
patience_counter = 0

for epoch in range(n_epochs):
    # Train
    policy.train()
    train_loss = 0
    train_batches = 0
    
    for batch_obs, batch_act, batch_masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs}", leave=False):
        batch_obs = batch_obs.to(device)
        batch_act = batch_act.to(device)
        
        batch_masks_device = {
            'r1_mask': batch_masks['r1_mask'].to(device),
            'c1_masks': batch_masks['c1_masks'].to(device),
            'r2_masks': batch_masks['r2_masks'].to(device),
            'c2_masks': batch_masks['c2_masks'].to(device)
        }
        
        action_tuple = tuple(batch_act[:, i] for i in range(4))
        _, log_prob, _, _ = policy(batch_obs, action=action_tuple, masks=batch_masks_device)
        
        loss = -log_prob.mean()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_batches += 1
    
    train_loss /= train_batches
    
    # Val
    policy.eval()
    val_loss = 0
    val_batches = 0
    
    with torch.no_grad():
        for batch_obs, batch_act, batch_masks in val_loader:
            batch_obs = batch_obs.to(device)
            batch_act = batch_act.to(device)
            
            batch_masks_device = {
                'r1_mask': batch_masks['r1_mask'].to(device),
                'c1_masks': batch_masks['c1_masks'].to(device),
                'r2_masks': batch_masks['r2_masks'].to(device),
                'c2_masks': batch_masks['c2_masks'].to(device)
            }
            
            action_tuple = tuple(batch_act[:, i] for i in range(4))
            _, log_prob, _, _ = policy(batch_obs, action=action_tuple, masks=batch_masks_device)
            
            loss = -log_prob.mean()
            val_loss += loss.item()
            val_batches += 1
    
    val_loss /= val_batches
    
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
    
    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(policy.state_dict(), 'bc_policy_best_v3.pt')
        print(f"  ✅ Best model saved (Val Loss: {val_loss:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  ⏳ No improvement for {patience_counter} epochs")
        
        if patience_counter >= patience:
            print(f"\n🛑 Early stopping at epoch {epoch+1}")
            break

print("\n✅ Behavior Cloning 완료!")
print(f"Best Val Loss: {best_val_loss:.4f}")

## 🎯 3. 평가

In [ ]:
# Best 모델 로드
policy.load_state_dict(torch.load('bc_policy_best_v3.pt'))
policy.eval()

def evaluate_v3(policy, n_episodes=50, use_backward=True, target_coverage=0.95):
    episode_rewards = []
    illegal_counts = []
    
    with torch.no_grad():
        for i in tqdm(range(n_episodes), desc="Evaluating"):
            if use_backward:
                generator = BackwardBoardGenerator(rows=10, cols=17, seed=10000+i)
                board, _ = generator.generate(target_coverage=target_coverage)
                wrapped_env = make_autoregressive_env(rows=10, cols=17)
                env = wrapped_env.env
                env.board = board.astype(np.int16)
                obs = board.clip(0, 9).astype(np.int8)
            else:
                wrapped_env = make_autoregressive_env(rows=10, cols=17)
                obs, info = wrapped_env.reset(seed=10000+i)
            
            episode_reward = 0
            steps = 0
            illegal_count = 0
            
            while True:
                masks_np = wrapped_env.get_autoregressive_masks()
                masks_torch = {
                    'r1_mask': torch.from_numpy(masks_np['r1_mask']).to(device),
                    'c1_masks': torch.from_numpy(masks_np['c1_masks']).to(device),
                    'r2_masks': torch.from_numpy(masks_np['r2_masks']).to(device),
                    'c2_masks': torch.from_numpy(masks_np['c2_masks']).to(device)
                }
                
                obs_tensor = torch.from_numpy(obs).float().unsqueeze(0).unsqueeze(0).to(device)
                action_tuple, _, _, _ = policy(obs_tensor, deterministic=True, masks=masks_torch)
                
                r1 = int(action_tuple[0][0].item())
                c1 = int(action_tuple[1][0].item())
                r2 = int(action_tuple[2][0].item())
                c2 = int(action_tuple[3][0].item())
                
                obs, reward, terminated, truncated, info = wrapped_env.step_with_coords(r1, c1, r2, c2)
                
                if info.get('illegal_action', False):
                    illegal_count += 1
                
                episode_reward += reward
                steps += 1
                
                if terminated or truncated or steps >= 500:
                    break
            
            episode_rewards.append(episode_reward)
            illegal_counts.append(illegal_count)
    
    return episode_rewards, illegal_counts

In [ ]:
# 평가 (역방향 생성 보드)
print("\n=== 95% 제거 가능 보드 평가 ===")
results_95, illegal_95 = evaluate_v3(policy, n_episodes=50, use_backward=True, target_coverage=0.95)
print(f"평균: {np.mean(results_95):.1f} ± {np.std(results_95):.1f}")
print(f"최대: {max(results_95):.0f}/170 ({max(results_95)/170*100:.1f}%)")
print(f"범위: [{min(results_95):.0f}, {max(results_95):.0f}]")
print(f"평균 불법 행동: {np.mean(illegal_95):.2f}회")

# 평가 (일반 보드)
print("\n=== 일반 보드 평가 ===")
results_normal, illegal_normal = evaluate_v3(policy, n_episodes=50, use_backward=False)
print(f"평균: {np.mean(results_normal):.1f} ± {np.std(results_normal):.1f}")
print(f"최대: {max(results_normal):.0f}/170 ({max(results_normal)/170*100:.1f}%)")
print(f"평균 불법 행동: {np.mean(illegal_normal):.2f}회")

## 📊 4. Output 요약 (Claude Code용)

In [ ]:
print("\n" + "="*70)
print("🍎 AlphaApple V3 Training Summary (Claude Code용)")
print("="*70)
print()
print("[1] 데이터 수집 (Multiple Rollouts)")
print(f"  - Expert 데이터: {len(expert_data)} 에피소드")
print(f"  - 평균 보상: {np.mean([ep['total_reward'] for ep in expert_data]):.1f}")
print(f"  - 최대 보상: {max([ep['total_reward'] for ep in expert_data]):.0f}")
print(f"  - Rollouts per board: 5")
print(f"  - Temperature range: 0.5-2.0")
print()
print("[2] Training")
print(f"  - Best Val Loss: {best_val_loss:.4f}")
print(f"  - Early stopping 적용됨")
print()
print("[3] 평가 결과")
print(f"  95% 보드:  {np.mean(results_95):.1f}개 ({np.mean(results_95)/170*100:.1f}%)")
print(f"  일반 보드: {np.mean(results_normal):.1f}개 ({np.mean(results_normal)/170*100:.1f}%)")
print(f"  최대:      {max(results_95):.0f}개 ({max(results_95)/170*100:.1f}%)")
print()
print("[4] 버전 비교")
print("  | 버전 | 방법 | 평균 (95%) | 평균 (일반) | 최대 |")
print("  |------|------|-----------|------------|------|")
print("  | V1   | BC (no mask) | -500 (0%) | -500 (0%)  | 0    |")
print("  | V2   | BC + Mask | 101.5 (59.7%) | 104.2 (61.3%) | 129  |")
print(f"  | V3   | Multi-Rollout | {np.mean(results_95):.1f} ({np.mean(results_95)/170*100:.1f}%) | {np.mean(results_normal):.1f} ({np.mean(results_normal)/170*100:.1f}%) | {max(results_95):.0f}  |")
print()
print("[5] 목표 대비")
print(f"  사람 최고: 130개 (76.5%)")
print(f"  V3 최고:   {max(results_95):.0f}개 ({max(results_95)/170*100:.1f}%)")
print(f"  달성률:    {max(results_95)/130*100:.1f}%")
print()
print("[6] 다음 단계 제안")
if max(results_95) >= 130:
    print("  ✅ 사람 최고 달성! PPO fine-tuning이나 MCTS 시도")
elif max(results_95) >= 120:
    print("  🎯 거의 도달! Expert Iteration (V4) 시도")
else:
    print("  🔧 더 많은 rollouts (10개) 또는 beam search 필요")
print()
print("="*70)

## 💾 5. 모델 다운로드 (Colab)

In [ ]:
if IN_COLAB:
    from google.colab import files
    
    # 모델 다운로드
    files.download('bc_policy_best_v3.pt')
    print("✅ 모델 다운로드 완료")